# Import Library

In [68]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import classification_report as cr, accuracy_score as AS, precision_score as ps, recall_score as rs, mean_absolute_error as mae, mean_squared_error as mse, root_mean_squared_error as rmse, log_loss as ll
from sklearn.preprocessing import StandardScaler as ss, MinMaxScaler as mms, OneHotEncoder as ohe
from sklearn.ensemble import AdaBoostClassifier as abc, AdaBoostRegressor as abr
from sklearn.ensemble import GradientBoostingClassifier as gbc, GradientBoostingRegressor as gbr

from xgboost import XGBClassifier as xgb, XGBRegressor as xgbr

# Load and EDA DATASET

In [43]:
da = pd.read_csv(r"E:\DATA FOR TEST\Tele Customer churn\WA_Fn-UseC_-Telco-Customer-Churn _1.csv")
dat = da.copy()
dat["TotalCharges"] = pd.to_numeric(dat["TotalCharges"], errors = 'coerce')
dat["TotalCharges"] = dat["TotalCharges"].fillna(dat["TotalCharges"].mean())
dat.dropna(how = 'any', inplace = True)
data = dat.drop(columns = ['customerID', 'SeniorCitizen'])
print(data.isnull().sum())
# data.columns

gender              0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [44]:
# Split Data in x and y
x = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Feature Encoding
x = pd.get_dummies(x, columns = ['gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'], drop_first = True)

# Feature Encoding
y = y.map({'No':0, 'Yes' : 1})

# Split x and y in Training and Testing
x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.2, random_state = 42, shuffle = True)

# Feature Scaling
scaler  = ss()
x_train_sc = scaler.fit_transform(x_train)
x_test_sc = scaler.transform(x_test)

# Boosting Classifier

In [45]:
# Ada Boost Classifier Model Initilization 
ada_model = abc(n_estimators = 100, random_state = 42)  # Default n_estimator = 50

# Ada Boost Learning From Training Data
ada_model.fit(x_train_sc, y_train)

# Ada Boost Prediction on X Test
y_pred_ada = ada_model.predict(x_test_sc)

# Ada Boost Model Result
print(cr(y_test, y_pred_ada))

# Ada Boost Accuracy Score
print(round(AS(y_test, y_pred_ada) * 100, 2))

              precision    recall  f1-score   support

           0       0.81      0.92      0.86       967
           1       0.69      0.47      0.56       382

    accuracy                           0.79      1349
   macro avg       0.75      0.69      0.71      1349
weighted avg       0.78      0.79      0.78      1349

78.95


In [46]:
# Gradient Boosting Classifier Model Initilization
Gradient_model = gbc(n_estimators = 100, random_state = 42)

# Gradient Boosting Model learning from Training Data
Gradient_model.fit(x_train_sc, y_train)

# Gradient Boosting Model Prediction on X Test
y_pred_grad = Gradient_model.predict(x_test_sc)

# Gradient Boosting Classifier Results
print(cr(y_test, y_pred_grad))

# Accuracy Score
print(round(AS(y_test, y_pred_grad) * 100, 2))

              precision    recall  f1-score   support

           0       0.81      0.90      0.85       967
           1       0.65      0.48      0.55       382

    accuracy                           0.78      1349
   macro avg       0.73      0.69      0.70      1349
weighted avg       0.77      0.78      0.77      1349

77.98


In [47]:
# Xtreme Gradient Boosting Classifier Model Initilization
XGB = xgb()

# XGB Model Learning From Training Data
XGB.fit(x_train_sc, y_train)

# XGB Model Prediction on X Test Data
y_pred_xgb= XGB.predict(x_test_sc)

# XGB Results
print(cr(y_test, y_pred_xgb))

# XGB Accuracy Score
print(round(AS(y_test, y_pred_xgb) * 100, 2))

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       967
           1       0.64      0.51      0.57       382

    accuracy                           0.78      1349
   macro avg       0.73      0.70      0.71      1349
weighted avg       0.77      0.78      0.77      1349

77.84


In [62]:
# One Function KIDA
def all_boosting_model(x_train, x_test, y_train, y_test):
    models = {
        'Ada Boost Classifier': abc(n_estimators = 100,
                                   random_state = 42),
        
    'Gradient Boosting Classifier': gbc(n_estimators = 100,
                                       random_state = 42),
        
    'Xtreme Gradient Boost Classifier' : xgb(n_estimators = 100,
                                            random_state = 42)
    }
    for name, model in models.items():
        
        print("="*15, "*"*2, name, "*"*2, "="*15)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        print(f"Classification Report of {name}:\n {cr(y_test, y_pred)}")
        print(f"Accuracy Score Of {name} :\t {round(AS(y_test, y_pred)*100, 2)}")
        print()

all_boosting_model(x_train_sc, x_test_sc, y_train, y_test)

=============== ** Ada Boost Classifier ** ===============
Classification Report of Ada Boost Classifier:
               precision    recall  f1-score   support

           0       0.81      0.92      0.86       967
           1       0.69      0.47      0.56       382

    accuracy                           0.79      1349
   macro avg       0.75      0.69      0.71      1349
weighted avg       0.78      0.79      0.78      1349

Accuracy Score Of Ada Boost Classifier :	 78.95

=============== ** Gradient Boosting Classifier ** ===============
Classification Report of Gradient Boosting Classifier:
               precision    recall  f1-score   support

           0       0.81      0.90      0.86       967
           1       0.66      0.47      0.55       382

    accuracy                           0.78      1349
   macro avg       0.74      0.69      0.70      1349
weighted avg       0.77      0.78      0.77      1349

Accuracy Score Of Gradient Boosting Classifier :	 78.13

==========

# Boosting Regressor

In [73]:
# Ada Boost Classifier Model Initilization 
ada_model = abr(n_estimators = 100, random_state = 42)  # Default n_estimator = 50

# Ada Boost Learning From Training Data
ada_model.fit(x_train_sc, y_train)

# Ada Boost Prediction on X Test
y_pred_ada = ada_model.predict(x_test_sc)

# Ada Boost Mean Absolute Error
print("Mean Absolute Error : \t", round(mae(y_test, y_pred_ada) * 100, 2))

# Ada Boost Mean Squared Error
print("Mean Squared Error : \t", round(mse(y_test, y_pred_ada) * 100, 2))

# Ada Boost Root Mean Squared Error
print("Root Mean Squared Error : ", round(rmse(y_test, y_pred_ada) * 100, 2))

Mean Absolute Error : 	 33.42
Mean Squared Error : 	 15.28
Root Mean Squared Error :  39.09


In [ ]:
# Gradient Boosting Classifier Model Initilization
Gradient_model = gbc(n_estimators = 100, random_state = 42)

# Gradient Boosting Model learning from Training Data
Gradient_model.fit(x_train_sc, y_train)

# Gradient Boosting Model Prediction on X Test
y_pred_grad = Gradient_model.predict(x_test_sc)


# Gradient Boosting Mean Absolute Error
print("Mean Absolute Error : \t", round(mae(y_test, y_pred_grad) * 100, 2))

# Gradient Boosting Mean Squared Error
print("Mean Squared Error : \t", round(mse(y_test, y_pred_grad) * 100, 2))

# Gradient Boosting Root Mean Squared Error
print("Root Mean Squared Error : ", round(rmse(y_test, y_pred_grad) * 100, 2))

In [74]:
# Xtreme Gradient Boosting Classifier Model Initilization
XGB = xgb()

# XGB Model Learning From Training Data
XGB.fit(x_train_sc, y_train)

# XGB Model Prediction on X Test Data
y_pred_xgb= XGB.predict(x_test_sc)


# Xtreme Gradient Boost Mean Absolute Error
print("Mean Absolute Error : \t", round(mae(y_test, y_pred_xgb) * 100, 2))

# Xtreme Gradient Boost Mean Squared Error
print("Mean Squared Error : \t", round(mse(y_test, y_pred_xgb) * 100, 2))

# Xtreme Gradient Boost Root Mean Squared Error
print("Root Mean Squared Error : ", round(rmse(y_test, y_pred_xgb) * 100, 2))

Mean Absolute Error : 	 33.42
Mean Squared Error : 	 15.28
Root Mean Squared Error :  39.09


In [77]:
# One Function KIDA of Regression
def all_boosting_model(x_train, x_test, y_train, y_test):
    models = {
        'Ada Boost Regression': abc(n_estimators = 100,
                                   random_state = 42),
        
    'Gradient Boosting Regression': gbc(n_estimators = 100,
                                       random_state = 42),
        
    'Xtreme Gradient Boost Regression' : xgb(n_estimators = 100,
                                            random_state = 42)
    }
    for name, model in models.items():
        
        print("="*15, "*"*2, name, "*"*2, "="*15)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)

        # Mean Absolute Error
        print("Mean Absolute Error : \t", round(mae(y_test, y_pred) * 100, 2))
        
        # Mean Squared Error
        print("Mean Squared Error : \t", round(mse(y_test, y_pred) * 100, 2))
        
        # Root Mean Squared Error
        print("Root Mean Squared Error : ", round(rmse(y_test, y_pred) * 100, 2))
        print()

all_boosting_model(x_train_sc, x_test_sc, y_train, y_test)

=============== ** Ada Boost Regression ** ===============
Mean Absolute Error : 	 21.05
Mean Squared Error : 	 21.05
Root Mean Squared Error :  45.88

=============== ** Gradient Boosting Regression ** ===============
Mean Absolute Error : 	 21.87
Mean Squared Error : 	 21.87
Root Mean Squared Error :  46.76

=============== ** Xtreme Gradient Boost Regression ** ===============
Mean Absolute Error : 	 22.16
Mean Squared Error : 	 22.16
Root Mean Squared Error :  47.08

